# Hyperparameter grid

### Colab Setup

In [1]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    # mounting is optional: without it, results land in the Colab session
    # filesystem and are lost when the runtime ends
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        print(f"Drive not mounted. Results will be written to {ROOT} "
              "and lost when the session ends.")

Mounted at /content/drive


### Key Imports

In [2]:
import torch

from config import RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from experiments.grid import grid_search, winners

GRID_OUT = RESULTS_DIR / "grid.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("grid ->", GRID_OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

grid -> /content/drive/MyDrive/thesis/grid.csv | device: cuda
NVIDIA A100-SXM4-40GB


## Six encoders, 3 learning rates x 3 batch sizes

In [3]:
grid_search(GRID_OUT, models=list(SHAH_PLM), seeds=SHAH_SEEDS, device=DEVICE)

resuming -- 162 configs already done
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  

model,seed,lr,batch_size,epochs,val_ce,val_acc,val_f1,val_macro_f1
str,i64,f64,i64,i64,f64,f64,f64,f64
"""bert-base-uncased""",5768,0.00002,32,22,1.6862,0.6869,0.6848,0.6382
"""bert-base-uncased""",5768,0.00001,32,14,0.9568,0.6465,0.6555,0.6196
"""bert-base-uncased""",5768,0.000005,32,25,1.1784,0.6591,0.6622,0.6222
"""bert-base-uncased""",5768,0.00002,16,19,1.6403,0.6919,0.6931,0.6528
"""bert-base-uncased""",5768,0.00001,16,15,1.2192,0.6818,0.6836,0.6482
…,…,…,…,…,…,…,…,…
"""flang-roberta""",944601,0.00001,16,17,1.1868,0.6944,0.6965,0.6698
"""flang-roberta""",944601,0.000005,16,16,0.974,0.6818,0.6848,0.656
"""flang-roberta""",944601,0.00002,8,24,1.5336,0.7323,0.7302,0.7081


## Winners

In [4]:
import polars as pl

print(winners(pl.read_csv(GRID_OUT)))

shape: (6, 7)
┌──────────────────┬─────────┬────────────┬──────────────────┬─────────────────┬─────────────┬─────┐
│ model            ┆ lr      ┆ batch_size ┆ mean_val_macro_f ┆ std_val_macro_f ┆ mean_epochs ┆ n   │
│ ---              ┆ ---     ┆ ---        ┆ 1                ┆ 1               ┆ ---         ┆ --- │
│ str              ┆ f64     ┆ i64        ┆ ---              ┆ ---             ┆ f64         ┆ u32 │
│                  ┆         ┆            ┆ f64              ┆ f64             ┆             ┆     │
╞══════════════════╪═════════╪════════════╪══════════════════╪═════════════════╪═════════════╪═════╡
│ roberta-large    ┆ 0.00001 ┆ 16         ┆ 0.735833         ┆ 0.018911        ┆ 20.0        ┆ 3   │
│ roberta-base     ┆ 0.00001 ┆ 32         ┆ 0.6899           ┆ 0.011233        ┆ 22.333333   ┆ 3   │
│ bert-large-uncas ┆ 0.00001 ┆ 8          ┆ 0.6753           ┆ 0.019613        ┆ 18.666667   ┆ 3   │
│ ed               ┆         ┆            ┆                  ┆               